# 03 · Model Training, Time Validation and Prediction

This notebook uses the derived files from Notebook 02. It compares five models,
their cumulative feature tiers, optional Optuna searches, and independent feature-group combinations.

**Default information policy: rolling monthly macro inputs.** A rental record in month t
uses macroeconomic observations from t−1 in both fitting and prediction. The model is fitted
using the specified historical rental labels; later validation, holdout, and Kaggle rental
labels are never used to update it during prediction.

**Time design**: fit on 2021–2022, select the configuration using 2023, evaluate on
2024–2025-03, and refit on all labeled records for the Kaggle test set. The holdout has already
been inspected during policy revision, so subsequent comparisons on it are diagnostics,
not an untouched estimate of generalization.

Run the notebook from the top after changing the macro policy. All stale execution outputs
from the uploaded fixed-origin run have been cleared. Rolling and fixed runs use separate
output, model, and submission directories. QUICK_MODE remains an execution check only.

In [ ]:
from pathlib import Path
import os
import json
import re
import hashlib
import platform
import importlib.metadata as package_metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from IPython.display import display

SEED = 42
rng = np.random.default_rng(SEED)
pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid', context='notebook')

def locate_project_root():
    override = os.environ.get('HDB_PROJECT_ROOT')
    candidates = [Path(override).expanduser().resolve()] if override else [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / 'data' / 'raw' / 'train.csv').is_file():
            return candidate
    raise FileNotFoundError('Put train.csv and test.csv in PROJECT/data/raw/. Open this notebook from PROJECT or PROJECT/notebooks/.')

ROOT = locate_project_root()
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
OUTPUTS = ROOT / 'outputs'
for directory in [PROCESSED, OUTPUTS]:
    directory.mkdir(parents=True, exist_ok=True)

FILES = ['train.csv', 'test.csv', 'sg-hdb-block.csv', 'sg-mrt-stations.csv',
         'sg-shopping-malls.csv', 'sg-schools.csv', 'sg-coe-prices.csv', 'sg-stock-prices.csv']

def raw_path(name):
    candidates = [RAW / name, RAW / 'auxiliary-data' / name]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f'Missing {name}: expected in {RAW} or {RAW / "auxiliary-data"}.')

def load_raw(name):
    # Preserve identifiers such as 090034 and 321A as text.
    return pd.read_csv(raw_path(name), dtype={'BLOCK': 'string', 'POSTAL_CODE': 'string'})

def clean_text(series):
    return series.astype('string').str.normalize('NFKC').str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)

def canonical_flat_type(series):
    return clean_text(series).str.replace(r'[\s-]*room\b', '-room', regex=True)

def canonical_keys(frame):
    out = frame.copy()
    for column in ['TOWN', 'BLOCK', 'STREET']:
        if column in out:
            out[column] = clean_text(out[column])
    if 'FLAT_TYPE' in out:
        out['FLAT_TYPE'] = canonical_flat_type(out['FLAT_TYPE'])
    return out

def parse_amount(series):
    return pd.to_numeric(series.astype('string').str.replace(r'[$,\s]', '', regex=True), errors='coerce')

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

print('Project:', ROOT)
print('Python:', platform.python_version())


## 1. Experiment Configuration

**RMSE** is used for model selection by default, while MAE, RMSLE, and R² are also reported.


In [ ]:
import time
import copy
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_squared_log_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

QUICK_MODE = os.environ.get('HDB_QUICK', '0') == '1'   # Default: full data
PRIMARY_METRIC = 'RMSE'  # RMSE, MAE or RMSLE
MACRO_POLICY = 'rolling_monthly'  # Each prediction month uses its preceding month's macro data.
VALID_START = pd.Timestamp('2023-01-01')
HOLDOUT_START = pd.Timestamp('2024-01-01')
N_JOBS = min(8, os.cpu_count() or 1)
RUN_ABLATION = True
RUN_OPTUNA = True
OPTUNA_TRIALS = 10
RUN_ROLLING_BACKTEST = False
RUN_FAST_MACRO_CHECK = True  # Existing Ridge configuration, before expensive searches.
RUN_MACRO_POLICY_COMPARISON = True  # Same fitted model, two holdout macro-input policies.
RUN_SHAP = False
RUN_GROUP_COMBINATIONS = True
GROUP_COMBO_USE_BEST_PARAMS = True  # Freeze the better default/tuned candidate's parameters.
GROUP_COMBO_REQUIRE_CORE = False   # False: 15 combinations; True: 8 containing core.
SUBMISSION_TARGET_COLUMN = None  # Set only if the real template has an ambiguous target name
assert PRIMARY_METRIC in ['RMSE', 'MAE', 'RMSLE']
assert MACRO_POLICY in ['fixed_origin', 'rolling_monthly']

RUN_NAME = ('quick' if QUICK_MODE else 'full') + '_' + MACRO_POLICY
RUN_DIR = OUTPUTS / 'training' / RUN_NAME
MODEL_DIR = RUN_DIR / 'models'
SUBMISSION_DIR = ROOT / 'submissions' / RUN_NAME
for directory in [RUN_DIR, MODEL_DIR, SUBMISSION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
print('Mode:', RUN_NAME, '| Threads:', N_JOBS, '| Selection metric:', PRIMARY_METRIC)

## 2. Loading Features and Verifying Provenance and Row Order

If the raw CSV files have changed, rerun feature engineering. Outdated derived features must not be used.


In [ ]:
metadata_path = PROCESSED / 'feature_metadata.json'
if not metadata_path.is_file():
    raise FileNotFoundError('Run 02_feature_engineering.ipynb first.')
feature_meta = json.loads(metadata_path.read_text(encoding='utf-8'))
for name, expected_hash in feature_meta['source_sha256'].items():
    assert file_sha256(raw_path(name)) == expected_hash, f'{name} changed: rerun Notebook 02.'
cat_columns = feature_meta['categorical_features']
read_types = {column: 'string' for column in cat_columns}
train_features = pd.read_csv(PROCESSED / 'train_features.csv', dtype=read_types, parse_dates=['rent_approval_date'])
test_features = pd.read_csv(PROCESSED / 'test_features.csv', dtype=read_types, parse_dates=['rent_approval_date'])
targets = pd.read_csv(PROCESSED / 'train_targets.csv')
macro = pd.read_csv(PROCESSED / 'macro_monthly.csv', parse_dates=['source_month']).set_index('source_month').sort_index()
assert np.array_equal(train_features.row_id, targets.row_id)
assert train_features.row_id.is_unique and test_features.row_id.is_unique
assert list(train_features.columns) == list(test_features.columns)
assert 'monthly_rent' not in train_features
y = targets.set_index('row_id').monthly_rent
feature_sets = feature_meta['feature_sets']
print('Rows:', len(train_features), 'train /', len(test_features), 'test')


### Macro data coverage audit

Checks raw-source availability, processed months, and historical return warm-up gaps.

In [ ]:
# Verify raw-source and processed-feature coverage before training any model.
month_map = {name: month for month, name in enumerate(
    ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec'], 1
)}
coe_audit = load_raw('sg-coe-prices.csv').copy()
coe_month_number = clean_text(coe_audit['MONTH']).map(month_map)
if coe_month_number.isna().any():
    raise ValueError('Unrecognized COE month names; inspect sg-coe-prices.csv.')
coe_audit['source_month'] = pd.to_datetime({
    'year': coe_audit['YEAR'], 'month': coe_month_number, 'day': 1
})
coe_audit['category'] = clean_text(coe_audit['CATEGORY'])
coe_audit['premium'] = parse_amount(coe_audit['QUOTA_PREMIUM'])
coe_counts = coe_audit.dropna(subset=['premium']).groupby(
    ['source_month', 'category']
).size().unstack(fill_value=0).reindex(columns=['a', 'b', 'c', 'e'], fill_value=0)

stock_audit = load_raw('sg-stock-prices.csv').copy()
stock_audit['DATE'] = pd.to_datetime(stock_audit['DATE'], errors='raise')
stock_audit['source_month'] = stock_audit['DATE'].dt.to_period('M').dt.to_timestamp()
audit_symbols = feature_meta.get('selected_stock_symbols',
    ['D05.SI', 'O39.SI', 'U11.SI', 'C6L.SI', 'Z74.SI', 'C38U.SI'])
stock_counts = stock_audit[
    stock_audit['SYMBOL'].isin(audit_symbols) & stock_audit['CLOSE'].notna()
].groupby(['source_month', 'SYMBOL']).size().unstack(fill_value=0).reindex(
    columns=audit_symbols, fill_value=0
)

macro_columns = list(feature_meta['macro_columns'])
assert macro.index.is_unique and macro.index.is_monotonic_increasing
assert macro.index.equals(macro.index.to_period('M').to_timestamp())
assert set(macro_columns).issubset(macro.columns)
initial_missing_month = macro.index.min() - pd.offsets.MonthBegin(1)
return_warmup_end = macro.index.min() + pd.DateOffset(months=3)
coverage_frames = {
    'train': train_features,
    'validation': train_features[
        (train_features.rent_approval_date >= VALID_START) &
        (train_features.rent_approval_date < HOLDOUT_START)],
    'holdout': train_features[train_features.rent_approval_date >= HOLDOUT_START],
    'test': test_features
}
coverage_rows = []
for split, frame in coverage_frames.items():
    required = pd.DatetimeIndex(
        frame.rent_approval_date - pd.offsets.MonthBegin(1)
    ).unique().sort_values()
    present = required.intersection(macro.index)
    absent = required.difference(macro.index)
    unexpected_absent = absent.difference(pd.DatetimeIndex([initial_missing_month]))
    if len(unexpected_absent):
        raise ValueError(f'{split}: missing processed macro months '
            f'{unexpected_absent.strftime("%Y-%m").tolist()}; rerun Notebook 02.')
    missing_coe = coe_counts.reindex(present).fillna(0).eq(0)
    missing_stocks = stock_counts.reindex(present).fillna(0).eq(0)
    complete_history = present[present >= return_warmup_end]
    later_missing_cells = int(macro.loc[complete_history, macro_columns].isna().sum().sum())
    if missing_coe.to_numpy().any() or missing_stocks.to_numpy().any() or later_missing_cells:
        raise ValueError(f'{split}: incomplete COE, stock, or processed macro data; inspect raw files and rerun Notebook 02.')
    coverage_rows.append({
        'split': split, 'rows': len(frame),
        'requested_source_first': required.min().strftime('%Y-%m'),
        'requested_source_last': required.max().strftime('%Y-%m'),
        'source_months_requested': len(required),
        'missing_months_after_initial_boundary': len(unexpected_absent),
        'initial_boundary_missing_months': ','.join(absent.strftime('%Y-%m')),
        'missing_cells_after_return_warmup': later_missing_cells
    })
macro_coverage_audit = pd.DataFrame(coverage_rows)
macro_coverage_audit.to_csv(RUN_DIR / 'macro_coverage_audit.csv', index=False)
display(macro_coverage_audit)
print('Raw COE source months:', coe_counts.index.min(), 'to', coe_counts.index.max())
print('Selected-stock source months:', stock_counts.index.min(), 'to', stock_counts.index.max())
print('Initial 2021 rows may lack December 2020 macro data; initial 1m/3m returns also need history.')
print('These historical warm-up gaps remain missing and are imputed from fitting data, never from future months.')

## 3. Aligning Macroeconomic Information by Prediction Month

The default `rolling_monthly` policy uses the immediately preceding month for every
fitting and prediction row. For example, January 2024 uses December 2023, February
2024 uses January 2024, and December 2024 uses November 2024.

The supplied COE and selected stock files cover all of 2024 and all preceding months
needed by the Kaggle test rows. The preceding audit verifies the local files and the
processed macro table rather than silently filling missing future months.

This evaluates the information condition in which preceding-month macro observations
are available when the rental month is predicted. The source files do not contain
publication timestamps, so month t−1 availability is an explicit assumption.

`fixed_origin` remains available for a separate scenario in which all later predictions
must be issued with information frozen at the training endpoint. Changing policies
does not introduce any future rental labels into fitting.

In [ ]:
def attach_macro(frame, origin_month=None, policy=None):
    policy = MACRO_POLICY if policy is None else policy
    if policy not in ['fixed_origin', 'rolling_monthly']:
        raise ValueError(f'Unknown macro policy: {policy}')
    out = frame.copy()
    approval_month = pd.to_datetime(out.rent_approval_date, errors='raise').dt.to_period('M').dt.to_timestamp()
    requested = approval_month - pd.offsets.MonthBegin(1)
    if policy == 'fixed_origin':
        if origin_month is None:
            raise ValueError('fixed_origin requires origin_month.')
        origin = pd.Timestamp(origin_month).to_period('M').to_timestamp()
        requested = requested.where(requested <= origin, origin)

    requested_index = pd.DatetimeIndex(requested)
    missing = requested_index.unique().difference(macro.index)
    allowed_initial_gap = pd.DatetimeIndex([macro.index.min() - pd.offsets.MonthBegin(1)])
    unexpected_missing = missing.difference(allowed_initial_gap)
    if len(unexpected_missing):
        raise ValueError('Missing macro source months: '
            f'{unexpected_missing.strftime("%Y-%m").tolist()}. Check coverage and rerun Notebook 02.')
    aligned = macro.reindex(requested_index).reset_index(drop=True)
    out = pd.concat([out.reset_index(drop=True), aligned], axis=1)
    out['macro_source_month'] = requested_index
    assert out.macro_source_month.lt(out.rent_approval_date).all()
    if policy == 'rolling_monthly':
        assert np.array_equal(out.macro_source_month.to_numpy(),
                              (approval_month - pd.offsets.MonthBegin(1)).to_numpy())
    else:
        assert out.macro_source_month.le(origin).all()
    return out

def cap_rows(frame, limit):
    return frame.sample(min(limit, len(frame)), random_state=SEED).sort_values('row_id').reset_index(drop=True)

fit_raw = train_features[train_features.rent_approval_date < VALID_START].copy()
valid_raw = train_features[(train_features.rent_approval_date >= VALID_START) &
                           (train_features.rent_approval_date < HOLDOUT_START)].copy()
holdout_raw = train_features[train_features.rent_approval_date >= HOLDOUT_START].copy()
assert len(fit_raw) and len(valid_raw) and len(holdout_raw)
assert fit_raw.rent_approval_date.max() < valid_raw.rent_approval_date.min()
assert valid_raw.rent_approval_date.max() < holdout_raw.rent_approval_date.min()
if QUICK_MODE:
    fit_raw = cap_rows(fit_raw, 12000)
    valid_raw = cap_rows(valid_raw, 5000)
    holdout_raw = cap_rows(holdout_raw, 5000)

tuning_origin = fit_raw.rent_approval_date.max()
fit_data = attach_macro(fit_raw, tuning_origin)
valid_data = attach_macro(valid_raw, tuning_origin)
y_fit = y.loc[fit_data.row_id].to_numpy()
y_valid = y.loc[valid_data.row_id].to_numpy()
split_summary = pd.DataFrame([
    {'split': name, 'rows_used': len(frame), 'first_month': frame.rent_approval_date.min().strftime('%Y-%m'),
     'last_month': frame.rent_approval_date.max().strftime('%Y-%m')}
    for name, frame in [('fit', fit_raw), ('validation', valid_raw), ('independent_holdout', holdout_raw)]
])
display(split_summary)
print('Validation macro source month maximum:', valid_data.macro_source_month.max())

## 4. Within-fold Preprocessing, Model Construction, and Evaluation Functions

- Ridge / Random Forest / XGBoost / LightGBM: Imputation is fitted within the training fold, categorical variables are one-hot encoded, and rare categories are grouped using `min_frequency`.
- CatBoost: Categorical meaning is preserved rather than arbitrarily encoding categories such as Town as continuous integers. Categorical feature indices are generated from the fixed column order after preprocessing.
- All candidates use the same data splits, target scale, and evaluation functions. Negative predictions are clipped to 0 consistently; predictions are not clipped using the observed range of true rents.
- Fixed tree counts are used for reproducibility. This version does not use the independent holdout for early stopping.


In [ ]:
def prepared_inputs(frame, columns):
    out = frame[columns].copy()
    for column in columns:
        if column in cat_columns:
            out[column] = out[column].astype('string').fillna('__MISSING__').astype(str)
        else:
            out[column] = pd.to_numeric(out[column], errors='coerce').replace([np.inf, -np.inf], np.nan)
    return out

def build_pipeline(kind, columns, params=None):
    cats = [column for column in columns if column in cat_columns]
    nums = [column for column in columns if column not in cats]
    if kind == 'catboost':
        preprocessor = ColumnTransformer([
            ('num', SimpleImputer(strategy='median', keep_empty_features=True), nums),
            ('cat', 'passthrough', cats)
        ], sparse_threshold=0)
        defaults = dict(iterations=80 if QUICK_MODE else 400, depth=6, learning_rate=.05,
                        loss_function='RMSE', random_seed=SEED, thread_count=N_JOBS,
                        verbose=False, allow_writing_files=False, cat_features=list(range(len(nums), len(columns))))
        defaults.update(params or {})
        estimator = CatBoostRegressor(**defaults)
    else:
        numeric_pipe = Pipeline([
            ('impute', SimpleImputer(strategy='median', keep_empty_features=True)),
            ('scale', StandardScaler(with_mean=False))
        ])
        preprocessor = ColumnTransformer([
            ('num', numeric_pipe, nums),
            ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=10, sparse_output=True), cats)
        ], sparse_threshold=1.0)
        if kind == 'ridge':
            defaults = dict(alpha=100., solver='lsqr', tol=1e-4)
            defaults.update(params or {}); estimator = Ridge(**defaults)
        elif kind == 'random_forest':
            defaults = dict(n_estimators=30 if QUICK_MODE else 100, max_depth=16, min_samples_leaf=8,
                            max_features=.7, n_jobs=N_JOBS, random_state=SEED)
            defaults.update(params or {}); estimator = RandomForestRegressor(**defaults)
        elif kind == 'xgboost':
            defaults = dict(n_estimators=80 if QUICK_MODE else 400, max_depth=6, learning_rate=.05,
                            subsample=.8, colsample_bytree=.8, min_child_weight=10, reg_lambda=5,
                            objective='reg:squarederror', tree_method='hist', n_jobs=N_JOBS, random_state=SEED)
            defaults.update(params or {}); estimator = XGBRegressor(**defaults)
        elif kind == 'lightgbm':
            defaults = dict(n_estimators=80 if QUICK_MODE else 400, num_leaves=31, learning_rate=.05,
                            min_child_samples=30, colsample_bytree=.8, subsample=.8, subsample_freq=1,
                            reg_lambda=5, n_jobs=N_JOBS, random_state=SEED, verbosity=-1)
            defaults.update(params or {}); estimator = LGBMRegressor(**defaults)
        else:
            raise ValueError(kind)
    return Pipeline([('preprocess', preprocessor), ('model', estimator)])

def fit_bundle(spec, frame, labels):
    columns = list(feature_sets[spec['feature_set']])
    # Avoid very wide address one-hot matrices in the Random Forest baseline.
    # Geographic coordinates and region/subzone information still describe location.
    if spec['kind'] == 'random_forest':
        columns = [c for c in columns if c not in ['building_id', 'street']]
    bundle = {'spec': copy.deepcopy(spec), 'columns': columns}
    if spec['kind'] == 'global_mean':
        bundle['global_mean'] = float(np.mean(labels))
    elif spec['kind'] == 'town_flat_mean':
        history = frame[['town', 'flat_type']].copy()
        history['target'] = np.asarray(labels)
        mean = float(np.mean(labels))
        grouped = history.groupby(['town', 'flat_type']).target.agg(['mean', 'count'])
        grouped['smoothed'] = (grouped['count'] * grouped['mean'] + 20 * mean) / (grouped['count'] + 20)
        bundle.update(global_mean=mean, group_means=grouped.smoothed.to_dict())
    else:
        pipe = build_pipeline(spec['kind'], columns, spec.get('params'))
        pipe.fit(prepared_inputs(frame, columns), labels)
        bundle['pipeline'] = pipe
    return bundle

def predict_bundle(bundle, frame):
    kind = bundle['spec']['kind']
    if kind == 'global_mean':
        prediction = np.full(len(frame), bundle['global_mean'])
    elif kind == 'town_flat_mean':
        prediction = np.array([bundle['group_means'].get((town, flat), bundle['global_mean'])
                              for town, flat in zip(frame.town, frame.flat_type)])
    else:
        prediction = bundle['pipeline'].predict(prepared_inputs(frame, bundle['columns']))
    prediction = np.maximum(np.asarray(prediction, dtype=float), 0.)
    assert prediction.shape == (len(frame),) and np.isfinite(prediction).all()
    return prediction

def regression_metrics(actual, prediction):
    return {'RMSE': float(np.sqrt(mean_squared_error(actual, prediction))),
            'MAE': float(mean_absolute_error(actual, prediction)),
            'RMSLE': float(np.sqrt(mean_squared_log_error(actual, prediction))),
            'R2': float(r2_score(actual, prediction))}

def evaluate_spec(spec, fitting, labels, evaluation, actual):
    start = time.perf_counter()
    bundle = fit_bundle(spec, fitting, labels)
    prediction = predict_bundle(bundle, evaluation)
    row = {'model': spec['name'], 'kind': spec['kind'], 'feature_set': spec['feature_set'],
           **regression_metrics(actual, prediction), 'input_features': len(bundle['columns']), 'fit_predict_seconds': time.perf_counter() - start}
    return row, bundle, prediction


### Fast check before expensive model searches

Run through this cell first to compare the previous uploaded Ridge configuration under
both macro policies. Two Ridge fits cover the validation and already-inspected holdout
periods; no Optuna or feature-group search is required for this check. It is a diagnostic
of macro-input policy, not a new model-selection result.

In [ ]:
fast_macro_policy_results = pd.DataFrame()
if RUN_FAST_MACRO_CHECK:
    # The previous uploaded run selected Ridge(macro), alpha=4.185822729546974.
    # Keep that configuration fixed to isolate the effect of changing macro inputs.
    reference_spec = {
        'name': 'previous_ridge_macro_policy_check', 'kind': 'ridge',
        'feature_set': 'macro', 'params': {'alpha': 4.185822729546974}
    }
    fast_rows = []
    for split in ['validation', 'inspected_holdout_diagnostic']:
        if split == 'validation':
            reference_fit = fit_raw
            reference_eval = valid_raw
        else:
            reference_fit = train_features[
                train_features.rent_approval_date < HOLDOUT_START].copy()
            if QUICK_MODE:
                reference_fit = cap_rows(reference_fit, 18000)
            reference_eval = holdout_raw
        reference_origin = reference_fit.rent_approval_date.max()
        reference_fit_data = attach_macro(reference_fit, reference_origin, policy='rolling_monthly')
        pd.testing.assert_frame_equal(reference_fit_data,
            attach_macro(reference_fit, reference_origin, policy='fixed_origin'))
        reference_bundle = fit_bundle(reference_spec, reference_fit_data,
            y.loc[reference_fit_data.row_id].to_numpy())
        for policy in ['fixed_origin', 'rolling_monthly']:
            reference_eval_data = attach_macro(reference_eval, reference_origin, policy=policy)
            reference_prediction = predict_bundle(reference_bundle, reference_eval_data)
            fast_rows.append({
                'split': split, 'macro_policy': policy,
                **regression_metrics(y.loc[reference_eval_data.row_id].to_numpy(), reference_prediction),
                'macro_source_first': reference_eval_data.macro_source_month.min().strftime('%Y-%m'),
                'macro_source_last': reference_eval_data.macro_source_month.max().strftime('%Y-%m')
            })
        del reference_bundle, reference_prediction
    fast_macro_policy_results = pd.DataFrame(fast_rows)
    fast_macro_policy_results.to_csv(RUN_DIR / 'fast_macro_policy_check.csv', index=False)
    display(fast_macro_policy_results.round(4))
    print('Fixed previous configuration only; these results do not retune the rolling model.')

## 5. Four-tier Feature Ablation

The same configuration and parameters are used throughout, with feature groups added progressively to compare their incremental value. The best tier is selected solely using the 2023 validation score.


In [ ]:
MODEL_KINDS = ['ridge', 'random_forest', 'xgboost', 'lightgbm', 'catboost']
FEATURE_SET_NAMES = ['core', 'building', 'amenities', 'macro']

ablation_rows = []
best_feature_rows = []
selected_feature_sets = {kind: 'amenities' for kind in MODEL_KINDS}
ablation_results = pd.DataFrame()
best_features_by_model = pd.DataFrame()

if RUN_ABLATION:
    for kind in MODEL_KINDS:
        model_rows = []
        print(f'\nFeature ablation: {kind}')
        for feature_set in FEATURE_SET_NAMES:
            spec = {'name': f'{kind}_{feature_set}', 'kind': kind,
                    'feature_set': feature_set, 'params': {}}
            row, bundle, prediction = evaluate_spec(
                spec, fit_data, y_fit, valid_data, y_valid
            )
            model_rows.append(row)
            ablation_rows.append(row)
            print(f"  {feature_set}: {PRIMARY_METRIC}={row[PRIMARY_METRIC]:.3f}")
            del bundle, prediction

        model_results = pd.DataFrame(model_rows).sort_values(
            PRIMARY_METRIC, kind='stable'
        ).reset_index(drop=True)
        best = model_results.iloc[0]
        selected_feature_sets[kind] = str(best['feature_set'])
        best_feature_rows.append(best.to_dict())
        model_results.to_csv(RUN_DIR / f'feature_ablation_{kind}.csv', index=False)
        display(model_results.round(4))

    ablation_results = pd.DataFrame(ablation_rows)
    best_features_by_model = pd.DataFrame(best_feature_rows).sort_values(
        PRIMARY_METRIC, kind='stable'
    ).reset_index(drop=True)
    ablation_results.to_csv(RUN_DIR / 'feature_ablation.csv', index=False)
    best_features_by_model.to_csv(RUN_DIR / 'best_features_by_model.csv', index=False)
    display(best_features_by_model.round(4))
else:
    print('Ablation skipped; using amenities for every model.')

print('Selected feature sets:', selected_feature_sets)


## 6. Baseline and Model Comparison

Candidates include the global mean, a smoothed Town × flat-type mean, Ridge, Random Forest, XGBoost, LightGBM, and CatBoost.
Group means are calculated entirely from rents in the fitting period. Unseen groups fall back to the global mean from that period.

To control training cost, Random Forest excludes the high-cardinality building_id / street features while retaining location features such as latitude/longitude, region, and postal district. This is an explicit model-specific input choice. Other models use all features in the selected tier, and the leaderboard records input_features.

Ridge can extrapolate along numerical time trends. Standard tree ensembles generally assign time values beyond the training range to existing leaves. Their limitations over long forecast horizons should therefore be discussed alongside the observed errors.


In [ ]:
specs = [
    {'name': kind, 'kind': kind, 'feature_set': 'core', 'params': {}}
    for kind in ['global_mean', 'town_flat_mean']
] + [
    {'name': kind, 'kind': kind,
     'feature_set': selected_feature_sets[kind], 'params': {}}
    for kind in MODEL_KINDS
]

result_rows, fitted_bundles, validation_predictions = [], {}, {}
for spec in specs:
    row, bundle, prediction = evaluate_spec(
        spec, fit_data, y_fit, valid_data, y_valid
    )
    result_rows.append(row)
    fitted_bundles[spec['name']] = bundle
    validation_predictions[spec['name']] = prediction
    print(f"{spec['name']} | {spec['feature_set']} | "
          f"{PRIMARY_METRIC}={row[PRIMARY_METRIC]:.3f}")

leaderboard = pd.DataFrame(result_rows).sort_values(
    PRIMARY_METRIC, kind='stable'
).reset_index(drop=True)
leaderboard.to_csv(RUN_DIR / 'validation_leaderboard.csv', index=False)
display(leaderboard.round(4))

## 7. Optional Optuna Tuning: Tuning-period Validation Only

When enabled, Optuna optimizes different models' depth, learning rate, tree count, regularization or other parameters.

In [ ]:
def suggest_model_params(trial, kind):
    if kind == 'ridge':
        return {'alpha': trial.suggest_float('alpha', 1e-2, 1e5, log=True)}

    if kind == 'random_forest':
        return {
            'n_estimators': trial.suggest_int(
                'n_estimators', 30 if QUICK_MODE else 100,
                150 if QUICK_MODE else 400, step=30 if QUICK_MODE else 100),
            'max_depth': trial.suggest_int('max_depth', 8, 24, step=4),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 2, 20),
            'max_features': trial.suggest_float('max_features', .4, 1.)
        }

    if kind not in ['xgboost', 'lightgbm', 'catboost']:
        raise ValueError(kind)

    iteration_key = 'iterations' if kind == 'catboost' else 'n_estimators'
    params = {
        iteration_key: trial.suggest_int(
            iteration_key, 80 if QUICK_MODE else 200,
            240 if QUICK_MODE else 800, step=40 if QUICK_MODE else 100),
        'learning_rate': trial.suggest_float('learning_rate', .02, .15, log=True)
    }

    if kind == 'catboost':
        params.update({
            'depth': trial.suggest_int('depth', 4, 8),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1., 20., log=True)
        })
        return params

    params.update({
        'subsample': trial.suggest_float('subsample', .6, 1.),
        'colsample_bytree': trial.suggest_float('colsample_bytree', .6, 1.),
        'reg_lambda': trial.suggest_float('reg_lambda', .1, 30., log=True)
    })
    if kind == 'xgboost':
        params.update({
            'max_depth': trial.suggest_int('max_depth', 3, 9),
            'min_child_weight': trial.suggest_float('min_child_weight', 1., 30., log=True)
        })
    else:
        params.update({
            'num_leaves': trial.suggest_categorical('num_leaves', [15, 31, 63]),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 80, step=10)
        })
    return params


def make_objective(kind, feature_set):
    def objective(trial):
        spec = {'name': f'{kind}_trial', 'kind': kind,
                'feature_set': feature_set,
                'params': suggest_model_params(trial, kind)}
        row, bundle, prediction = evaluate_spec(
            spec, fit_data, y_fit, valid_data, y_valid
        )
        del bundle, prediction
        return float(row[PRIMARY_METRIC])
    return objective


studies, tuned_specs, tuning_rows = {}, {}, []

# Remove previous tuned candidates when rerunning this cell.
tuned_names = {f'{kind}_optuna' for kind in MODEL_KINDS}
result_rows = [row for row in result_rows if row['model'] not in tuned_names]
for name in tuned_names:
    fitted_bundles.pop(name, None)
    validation_predictions.pop(name, None)

if RUN_OPTUNA:
    import optuna
    if OPTUNA_TRIALS < 1:
        raise ValueError('OPTUNA_TRIALS must be at least 1.')

    for kind in MODEL_KINDS:
        feature_set = selected_feature_sets[kind]
        print(f'\nOptuna: {kind} | features={feature_set}')
        study = optuna.create_study(
            direction='minimize',
            sampler=optuna.samplers.TPESampler(
                seed=SEED, n_startup_trials=min(5, max(1, OPTUNA_TRIALS // 3))
            )
        )
        study.optimize(
            make_objective(kind, feature_set),
            n_trials=OPTUNA_TRIALS, n_jobs=1, gc_after_trial=True
        )
        studies[kind] = study
        study.trials_dataframe().to_csv(
            RUN_DIR / f'optuna_trials_{kind}.csv', index=False
        )

        # Refit one best-trial candidate for each model.
        tuned_spec = {
            'name': f'{kind}_optuna', 'kind': kind,
            'feature_set': feature_set, 'params': dict(study.best_params)
        }
        tuned_specs[kind] = copy.deepcopy(tuned_spec)
        row, bundle, prediction = evaluate_spec(
            tuned_spec, fit_data, y_fit, valid_data, y_valid
        )
        result_rows.append(row)
        tuning_rows.append(row)
        fitted_bundles[tuned_spec['name']] = bundle
        validation_predictions[tuned_spec['name']] = prediction
        print('Best searched parameters:', study.best_params)
        print(f"{PRIMARY_METRIC}={row[PRIMARY_METRIC]:.3f}")

        # Checkpoint parameters after each model finishes.
        (RUN_DIR / 'optuna_best_specs.json').write_text(
            json.dumps(tuned_specs, indent=2), encoding='utf-8'
        )

    tuning_results = pd.DataFrame(tuning_rows).sort_values(
        PRIMARY_METRIC, kind='stable'
    ).reset_index(drop=True)
    tuning_results.to_csv(RUN_DIR / 'optuna_results.csv', index=False)
else:
    tuning_results = pd.DataFrame()
    print('Optuna skipped; set RUN_OPTUNA=True to enable.')

# Retain both default and tuned candidates in the final ranking.
leaderboard = pd.DataFrame(result_rows).sort_values(
    PRIMARY_METRIC, kind='stable'
).reset_index(drop=True)
leaderboard.to_csv(RUN_DIR / 'validation_leaderboard.csv', index=False)
display(leaderboard.round(4))


In [ ]:
# Existing feature_sets are cumulative. Recover four independent feature groups.
group_order = ['core', 'building', 'amenities', 'macro']
cumulative = {name: list(dict.fromkeys(feature_sets[name])) for name in group_order}
feature_groups = {}
seen = set()
for group in group_order:
    if not seen.issubset(cumulative[group]):
        raise ValueError(f'Expected cumulative feature sets; check {group}.')
    feature_groups[group] = [c for c in cumulative[group] if c not in seen]
    seen.update(cumulative[group])

if any(not columns for columns in feature_groups.values()):
    raise ValueError('Each of the four feature groups must contain at least one feature.')

group_combinations = {}
combination_members = {}
for size in range(1, len(group_order) + 1):
    for groups in combinations(group_order, size):
        if GROUP_COMBO_REQUIRE_CORE and 'core' not in groups:
            continue
        name = 'combo__' + '__'.join(groups)
        columns = list(dict.fromkeys(
            column for group in groups for column in feature_groups[group]
        ))
        group_combinations[name] = columns
        combination_members[name] = list(groups)

# fit_bundle looks up feature_sets[spec['feature_set']]. Keep these entries for refits.
feature_sets.update(group_combinations)
for frame in (fit_data, valid_data):
    missing = sorted(seen.difference(frame.columns))
    if missing:
        raise ValueError(f'Input frame is missing features: {missing}')

print('Independent group sizes:', {g: len(c) for g, c in feature_groups.items()})
print('Combinations per model:', len(group_combinations))

# Re-running this cell replaces only candidates belonging to this experiment.
candidate_names = {f'{kind}_group_combo' for kind in MODEL_KINDS}
result_rows = [row for row in result_rows if row['model'] not in candidate_names]
for name in candidate_names:
    fitted_bundles.pop(name, None)
    validation_predictions.pop(name, None)

group_combo_rows = []
group_combo_best_rows = []
group_combo_best_specs = {}
selected_group_feature_sets = {}

if RUN_GROUP_COMBINATIONS:
    for kind in MODEL_KINDS:
        # Choose parameters ONCE per model, then hold them fixed across all combinations.
        params = {}
        parameter_source = 'default'
        if GROUP_COMBO_USE_BEST_PARAMS:
            previous = [
                row for row in result_rows
                if row['kind'] == kind
                and row['model'] in {kind, f'{kind}_optuna'}
            ]
            if previous:
                best_previous = min(previous, key=lambda row: row[PRIMARY_METRIC])
                parameter_source = best_previous['model']
                params = copy.deepcopy(
                    fitted_bundles[parameter_source]['spec'].get('params', {})
                )

        print(f'\nGroup combinations: {kind} | parameters from {parameter_source}')
        model_rows = []
        for feature_set, columns in group_combinations.items():
            spec = {
                'name': f'{kind}__{feature_set}', 'kind': kind,
                'feature_set': feature_set, 'params': copy.deepcopy(params)
            }
            row, bundle, prediction = evaluate_spec(
                spec, fit_data, y_fit, valid_data, y_valid
            )
            row.update(
                groups=' + '.join(combination_members[feature_set]),
                parameter_source=parameter_source
            )
            model_rows.append(row)
            group_combo_rows.append(row)
            print(f"  {row['groups']}: {PRIMARY_METRIC}={row[PRIMARY_METRIC]:.3f}")
            del bundle, prediction  # Search phase retains scores, not fitted models.

        model_results = pd.DataFrame(model_rows).sort_values(
            PRIMARY_METRIC, kind='stable'
        ).reset_index(drop=True)
        model_results.to_csv(RUN_DIR / f'group_combinations_{kind}.csv', index=False)
        best = model_results.iloc[0]
        selected_group_feature_sets[kind] = str(best['feature_set'])

        # Refit exactly one winner for this model, matching the existing workflow.
        best_spec = {
            'name': f'{kind}_group_combo', 'kind': kind,
            'feature_set': str(best['feature_set']), 'params': copy.deepcopy(params)
        }
        row, bundle, prediction = evaluate_spec(
            best_spec, fit_data, y_fit, valid_data, y_valid
        )
        row.update(groups=str(best['groups']), parameter_source=parameter_source)
        result_rows.append(row)
        group_combo_best_rows.append(row)
        fitted_bundles[best_spec['name']] = bundle
        validation_predictions[best_spec['name']] = prediction
        group_combo_best_specs[kind] = copy.deepcopy(best_spec)
        print(f"Selected: {kind} | {row['groups']} | {PRIMARY_METRIC}={row[PRIMARY_METRIC]:.3f}")

    group_combo_results = pd.DataFrame(group_combo_rows).sort_values(
        ['kind', PRIMARY_METRIC], kind='stable'
    ).reset_index(drop=True)
    best_group_combinations = pd.DataFrame(group_combo_best_rows).sort_values(
        PRIMARY_METRIC, kind='stable'
    ).reset_index(drop=True)
    group_combo_results.to_csv(RUN_DIR / 'group_combinations_all.csv', index=False)
    best_group_combinations.to_csv(RUN_DIR / 'best_group_combinations.csv', index=False)

    # Persist column definitions as well as parameters, so custom feature-set names
    # can be reconstructed in a fresh session before calling fit_bundle again.
    (RUN_DIR / 'group_combinations_best_specs.json').write_text(
        json.dumps({
            'parameter_policy': 'best_existing_default_or_tuned' if GROUP_COMBO_USE_BEST_PARAMS else 'default',
            'require_core': GROUP_COMBO_REQUIRE_CORE,
            'feature_groups': feature_groups,
            'feature_sets': group_combinations,
            'best_specs': group_combo_best_specs
        }, indent=2), encoding='utf-8'
    )
    display(best_group_combinations.round(4))
else:
    group_combo_results = pd.DataFrame()
    best_group_combinations = pd.DataFrame()
    print('Feature-group combination experiment skipped.')

leaderboard = pd.DataFrame(result_rows).sort_values(
    PRIMARY_METRIC, kind='stable'
).reset_index(drop=True)
leaderboard.to_csv(RUN_DIR / 'validation_leaderboard.csv', index=False)
display(leaderboard.round(4))

# For the subsequent holdout/refit cell, preserve the winner's complete spec:
# best_name = leaderboard.iloc[0]['model']
# best_spec = copy.deepcopy(fitted_bundles[best_name]['spec'])

## 8. Freezing the Configuration and Examining Validation Errors

From this point onward, features, parameters, and the model must not be changed based on independent-holdout results. If further changes are made using those results, the holdout must be treated as another validation set rather than an independent performance estimate.

In [ ]:
leaderboard.to_csv(RUN_DIR / 'validation_leaderboard.csv', index=False)
winner_name = str(leaderboard.iloc[0].model)
winner_spec = copy.deepcopy(fitted_bundles[winner_name]['spec'])
winner_prediction = validation_predictions[winner_name]
print('Selected model:', winner_name, '| Feature set:', winner_spec['feature_set'])
print('Selection uses validation only, not the independent holdout.')
validation_report = valid_data[['row_id', 'rent_approval_date', 'town', 'flat_type', 'floor_area_sqm']].copy()
validation_report['actual'] = y_valid
validation_report['prediction'] = winner_prediction
validation_report['residual'] = validation_report.actual - validation_report.prediction
validation_report['absolute_error'] = validation_report.residual.abs()
validation_report.to_csv(RUN_DIR / 'validation_predictions.csv', index=False)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
leaderboard.set_index('model')[PRIMARY_METRIC].plot.barh(ax=axes[0], title=f'Validation {PRIMARY_METRIC}')
axes[1].scatter(y_valid, winner_prediction, s=5, alpha=.12)
lo, hi = float(y_valid.min()), float(y_valid.max())
axes[1].plot([lo, hi], [lo, hi], 'k--'); axes[1].set(xlabel='Actual rent', ylabel='Predicted rent', title=winner_name)
plt.tight_layout(); plt.show()
display(validation_report.groupby('town').agg(rows=('actual', 'size'), MAE=('absolute_error', 'mean'),
    mean_residual=('residual', 'mean')).sort_values('MAE', ascending=False).round(2))


## 9. Temporal Holdout Evaluation and Macro-policy Diagnostic

The configuration selected using 2023 is retrained on all labeled records before 2024
and evaluated on 2024–2025-03. Under the default rolling policy, each evaluation month
uses its own preceding month's macro observations.

The optional paired diagnostic reuses exactly the same fitted model, parameters,
features, and evaluation rows, changing only whether macro inputs are frozen at
2023-12 or updated monthly. It never selects a configuration from holdout scores.
Because this holdout has already informed policy revision, report it as a retrospective
diagnostic rather than a new untouched test.

In [ ]:
prehold_raw = train_features[train_features.rent_approval_date < HOLDOUT_START].copy()
if QUICK_MODE:
    prehold_raw = cap_rows(prehold_raw, 18000)
holdout_origin = prehold_raw.rent_approval_date.max()
prehold_data = attach_macro(prehold_raw, holdout_origin)
holdout_data = attach_macro(holdout_raw, holdout_origin)
y_prehold = y.loc[prehold_data.row_id].to_numpy()
y_holdout = y.loc[holdout_data.row_id].to_numpy()
holdout_row, holdout_bundle, holdout_prediction = evaluate_spec(
    winner_spec, prehold_data, y_prehold, holdout_data, y_holdout)
holdout_metrics = pd.DataFrame([{'evaluation': 'temporal_holdout_policy_revision', 'macro_policy': MACRO_POLICY, 'quick_mode': QUICK_MODE, **holdout_row}])
holdout_metrics.to_csv(RUN_DIR / 'holdout_metrics.csv', index=False)
display(holdout_metrics.round(4))
holdout_report = holdout_data[['row_id', 'rent_approval_date', 'town', 'flat_type', 'floor_area_sqm']].copy()
holdout_report['actual'] = y_holdout
holdout_report['prediction'] = holdout_prediction
holdout_report['residual'] = holdout_report.actual - holdout_report.prediction
holdout_report['absolute_error'] = holdout_report.residual.abs()
holdout_report.to_csv(RUN_DIR / 'holdout_predictions.csv', index=False)
holdout_by_month = holdout_report.groupby('rent_approval_date').agg(
    actual_mean=('actual', 'mean'), predicted_mean=('prediction', 'mean'), MAE=('absolute_error', 'mean'))
holdout_by_month.to_csv(RUN_DIR / 'holdout_by_month.csv')
holdout_by_month[['actual_mean', 'predicted_mean']].plot(figsize=(11, 4), title='Independent holdout: monthly average rent')
plt.ylabel('SGD/month'); plt.tight_layout(); plt.show()

In [ ]:
macro_policy_comparison = pd.DataFrame()
if RUN_MACRO_POLICY_COMPARISON:
    # Historical fitting rows precede the origin under either policy.
    pd.testing.assert_frame_equal(
        attach_macro(prehold_raw, holdout_origin, policy='fixed_origin'),
        attach_macro(prehold_raw, holdout_origin, policy='rolling_monthly')
    )
    policy_rows = []
    policy_monthly_rows = []
    for policy in ['fixed_origin', 'rolling_monthly']:
        policy_data = attach_macro(holdout_raw, holdout_origin, policy=policy)
        assert np.array_equal(policy_data.row_id.to_numpy(), holdout_data.row_id.to_numpy())
        policy_prediction = predict_bundle(holdout_bundle, policy_data)
        policy_rows.append({
            'macro_policy': policy, 'model': winner_spec['name'],
            'feature_set': winner_spec['feature_set'],
            **regression_metrics(y_holdout, policy_prediction),
            'macro_source_first': policy_data.macro_source_month.min().strftime('%Y-%m'),
            'macro_source_last': policy_data.macro_source_month.max().strftime('%Y-%m')
        })
        monthly = pd.DataFrame({
            'month': policy_data.rent_approval_date,
            'actual': y_holdout, 'prediction': policy_prediction
        })
        monthly['squared_error'] = (monthly.actual - monthly.prediction) ** 2
        by_month = monthly.groupby('month').agg(
            rows=('actual', 'size'), actual_mean=('actual', 'mean'),
            predicted_mean=('prediction', 'mean'), MSE=('squared_error', 'mean')
        ).reset_index()
        by_month['RMSE'] = np.sqrt(by_month.MSE)
        by_month['macro_policy'] = policy
        policy_monthly_rows.append(by_month)
    macro_policy_comparison = pd.DataFrame(policy_rows)
    macro_policy_comparison.to_csv(RUN_DIR / 'macro_policy_holdout_comparison.csv', index=False)
    pd.concat(policy_monthly_rows, ignore_index=True).to_csv(
        RUN_DIR / 'macro_policy_holdout_by_month.csv', index=False
    )
    display(macro_policy_comparison.round(4))
    print('Diagnostic only: the selected configuration is not changed using these holdout scores.')

## 10. Optional Supplementary Rolling-origin Backtests

The selected configuration remains fixed while several earlier forecast origins are examined using only pre-2024 data. The results are used to assess stability and do not automatically change the final configuration.

This is a retrospective diagnostic rather than a new independent test, because some labels from these periods have already been used during model selection.


In [ ]:
backtest_rows = []
if RUN_ROLLING_BACKTEST:
    for start in [pd.Timestamp('2022-07-01'), pd.Timestamp('2023-01-01'), pd.Timestamp('2023-07-01')]:
        end = min(start + pd.DateOffset(months=6), HOLDOUT_START)
        earlier = train_features[train_features.rent_approval_date < start].copy()
        later = train_features[(train_features.rent_approval_date >= start) & (train_features.rent_approval_date < end)].copy()
        if QUICK_MODE:
            earlier, later = cap_rows(earlier, 12000), cap_rows(later, 4000)
        origin = earlier.rent_approval_date.max()
        a, b = attach_macro(earlier, origin), attach_macro(later, origin)
        row, _, _ = evaluate_spec(winner_spec, a, y.loc[a.row_id].to_numpy(), b, y.loc[b.row_id].to_numpy())
        backtest_rows.append({'forecast_start': start.strftime('%Y-%m'), **row})
    pd.DataFrame(backtest_rows).to_csv(RUN_DIR / 'supplementary_backtest.csv', index=False)
    display(pd.DataFrame(backtest_rows).round(4))
else:
    print('Supplementary rolling-origin backtest skipped.')


## 11. Retraining on the Full Training Set, Saving the Model, and Generating Test Predictions

Once the model, features, and parameters are fixed, full mode retrains on all 150,000 labeled records. The former validation and holdout data may now be included in the final fit, but they cannot be reused to calculate a score claimed to be independent.

The original test-row order is preserved. In quick mode, the final fit still uses a sample, and both the output directories and metadata are marked as quick.

In [ ]:
final_raw = cap_rows(train_features, 25000) if QUICK_MODE else train_features.copy()
final_origin = train_features.rent_approval_date.max()
final_data = attach_macro(final_raw, final_origin)
test_data = attach_macro(test_features, final_origin)
final_bundle = fit_bundle(winner_spec, final_data, y.loc[final_data.row_id].to_numpy())
test_prediction = predict_bundle(final_bundle, test_data)
assert np.array_equal(test_data.row_id.to_numpy(), np.arange(len(test_features)))
prediction_table = pd.DataFrame({'row_id': test_data.row_id, 'MONTHLY_RENT': test_prediction})
prediction_table.to_csv(SUBMISSION_DIR / 'test_predictions.csv', index=False)
run_metadata = {
    'quick_mode': QUICK_MODE, 'selection_metric': PRIMARY_METRIC, 'macro_policy': MACRO_POLICY,
    'rental_training_end_month': final_origin.strftime('%Y-%m'),
    'forecast_information_cutoff_month': final_origin.strftime('%Y-%m') if MACRO_POLICY == 'fixed_origin' else None,
    'macro_lag_months': 1, 'rental_label_updates_during_prediction': False,
    'test_macro_source_first_month': test_data.macro_source_month.min().strftime('%Y-%m'),
    'test_macro_source_last_month': test_data.macro_source_month.max().strftime('%Y-%m'),
    'selected_model_spec': winner_spec, 'final_train_rows': len(final_data), 'test_rows': len(test_data),
    'duplicate_policy': 'retain', 'feature_pipeline_version': feature_meta['feature_pipeline_version'],
    'source_sha256': feature_meta['source_sha256'], 'validation_boundaries': {
        'valid_start': VALID_START.strftime('%Y-%m'), 'holdout_start': HOLDOUT_START.strftime('%Y-%m')},
    'python': platform.python_version(), 'package_versions': {
        p: package_metadata.version(p) for p in ['numpy', 'pandas', 'scikit-learn', 'catboost', 'xgboost', 'lightgbm']}
}
final_bundle['run_metadata'] = run_metadata
joblib.dump(final_bundle, MODEL_DIR / 'selected_model.joblib', compress=3)
(RUN_DIR / 'run_metadata.json').write_text(json.dumps(run_metadata, indent=2), encoding='utf-8')
split_summary.to_csv(RUN_DIR / 'split_summary.csv', index=False)
display(prediction_table.head())
display(prediction_table.MONTHLY_RENT.describe().to_frame())
print('Final training rows:', len(final_data), '| Test predictions:', len(prediction_table))
print('Saved model and preprocessing together:', MODEL_DIR / 'selected_model.joblib')

## 12. Feature Importance and Optional SHAP

Models fitted during the validation stage are used to interpret learned relationships, without treating explanations as causal conclusions. For models using one-hot encoding, feature importance is shown for the expanded columns. Ridge reports absolute coefficient values, which are not compared with tree-model importance values on the same scale.

Native feature importance measures can be biased. SHAP also explains model outputs rather than causal effects. SHAP is enabled only for supported tree models.


In [ ]:
explanation_bundle = fitted_bundles[winner_name]
if 'pipeline' in explanation_bundle:
    pipe = explanation_bundle['pipeline']
    estimator = pipe.named_steps['model']
    names = pipe.named_steps['preprocess'].get_feature_names_out()
    if hasattr(estimator, 'feature_importances_'):
        values = np.asarray(estimator.feature_importances_)
        importance_label = 'Model feature importance'
    elif hasattr(estimator, 'coef_'):
        values = np.abs(np.asarray(estimator.coef_).ravel())
        importance_label = 'Absolute coefficient (encoded/scaled inputs)'
    else:
        values = None
    if values is not None:
        importance = pd.DataFrame({'encoded_feature': names, 'importance': values}).sort_values('importance', ascending=False)
        importance.to_csv(RUN_DIR / 'feature_importance.csv', index=False)
        importance.head(20).iloc[::-1].plot.barh(x='encoded_feature', y='importance', figsize=(11, 7), legend=False)
        plt.title(importance_label); plt.tight_layout(); plt.show()
    if RUN_SHAP and winner_spec['kind'] in ['catboost', 'xgboost', 'lightgbm', 'random_forest']:
        import shap
        sample = valid_data.sample(min(300, len(valid_data)), random_state=SEED)
        matrix = pipe.named_steps['preprocess'].transform(prepared_inputs(sample, explanation_bundle['columns']))
        if hasattr(matrix, 'toarray'):
            matrix = matrix.toarray()
        explainer = shap.TreeExplainer(estimator)
        shap_values = explainer.shap_values(matrix)
        shap.summary_plot(shap_values, matrix, feature_names=names, max_display=20, show=False)
        plt.tight_layout(); plt.show()
    elif RUN_SHAP:
        print('SHAP skipped for the selected non-tree model.')
else:
    print('The selected baseline has no encoded feature-importance vector.')


## 13. Creating the Kaggle Submission

The competition requires the exact header `Id,Predicted`. IDs run from zero in the
original test-row order. No sample-submission file is required. Official scoring uses RMSE.

In [ ]:
# Prepare predictions.
prediction = np.asarray(test_prediction, dtype=float)
n_test = len(test_features)

assert prediction.shape == (n_test,), 'Prediction count or shape is incorrect.'
assert np.isfinite(prediction).all(), 'Predictions contain NaN or infinity.'
assert (prediction >= 0).all(), 'Predictions contain negative rent.'

# Confirm predictions follow the original test row order.
assert len(test_data) == n_test
assert np.array_equal(
    test_data['row_id'].to_numpy(),
    np.arange(n_test)
), 'Test rows are not in their original order.'

# Create the required Kaggle submission.
submission = pd.DataFrame({
    'Id': test_data['row_id'].to_numpy(dtype=np.int64),
    'Predicted': prediction
})

assert list(submission.columns) == ['Id', 'Predicted']
assert submission['Id'].is_unique

SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
submission_path = SUBMISSION_DIR / (
    'submission_QUICK.csv' if QUICK_MODE else 'submission.csv'
)
submission.to_csv(submission_path, index=False)

display(submission.head())
print('Submission rows:', len(submission))
print('Saved to:', submission_path)


## 14. Results and Limitations to Discuss in the Report

1. How much do the models improve over the global-mean and group-mean baselines during the 2023 validation period? How do results change across feature tiers?
2. Are errors consistent between the validation period and the 15-month holdout? Does prediction become more difficult as the forecast horizon increases?
3. Are the importance patterns for floor area, flat type, location, and amenities plausible? Which Town / flat-type groups have larger errors?
4. MRT / mall / school data are static snapshots without historical opening dates. MAX_FLOOR describes building height rather than the floor of the individual rental unit.
5. Furnishing and fee fields are constant in this dataset, so their effects cannot be estimated. Renovation, the unit's exact floor, orientation, interior condition, and other attributes are also not fully described.
6. Freezing economic information at a fixed forecast origin and incorporating newly available data each month represent different information conditions. The report must state which assumption is used.
7. Kaggle uses RMSE, matching PRIMARY_METRIC. Its hidden test error still cannot be calculated locally without true test rents. The inspected temporal holdout is diagnostic after policy revision.

API references: [scikit-learn OneHotEncoder](https://scikit-learn.org/1.6/modules/generated/sklearn.preprocessing.OneHotEncoder.html),
[SimpleImputer](https://scikit-learn.org/1.6/modules/generated/sklearn.impute.SimpleImputer.html),
[CatBoost](https://catboost.ai/docs/en/concepts/python-reference_catboostregressor),
[XGBoost](https://xgboost.readthedocs.io/en/release_3.0.0/python/python_api.html).